# OGAR Pipeline Demo

**What this notebook does:**  
Runs the full OGAR pipeline end-to-end — from world simulation through sensor readings, agent detection, and supervisor decision — so you can see exactly what each layer does.

```
WorldEngine  →  Sensors  →  Queue  →  ClusterAgent  →  SupervisorAgent
(ground truth)  (noisy)    (async)   (detect anomaly)  (decide action)
```

**Modes:**
- **Stub mode** (default): agents use deterministic logic, no LLM needed, runs instantly
- **LLM mode**: swap one cell below to wire in Claude or GPT-4o — agents reason with an actual LLM

Run cells top to bottom. Each section builds on the previous one.

## 0. Environment Setup

Load credentials and apply LangSmith tracing.  
Set `AI_ENV_FILE` to point at your secrets file before running.

In [ ]:
import os
import sys
import random
import logging

# ── Point at your secrets file ────────────────────────────────────────────────
# On K8s this env var is unset and credentials come from pod env vars instead.
os.environ.setdefault("AI_ENV_FILE", "/Users/chrislomeli/Source/SECRETS/.env")

# ── Load settings and apply LangSmith tracing ────────────────────────────────
from ogar.config import get_settings
settings = get_settings()
settings.apply_langsmith()   # sets LANGCHAIN_* env vars so LangGraph traces automatically

# ── Logging ───────────────────────────────────────────────────────────────────
# INFO shows what each agent node is doing.
# Set to WARNING if you want quieter output.
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(name)-35s  %(message)s",
    datefmt="%H:%M:%S",
)
# Suppress noisy third-party loggers
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("httpcore").setLevel(logging.WARNING)

print(f"LangSmith tracing: {settings.langchain_tracing_v2}")
print(f"LangSmith project: {settings.langchain_project}")
print(f"Anthropic key set: {bool(settings.anthropic_api_key)}")

## 1. Choose Your Mode

**Stub mode** (default below): no LLM, agents use placeholder logic, instant execution.  
**LLM mode**: uncomment one of the alternatives to wire in a real model.

Everything downstream uses `llm` — `None` means stub mode.

In [ ]:
llm = None   # stub mode — change this to enable LLM reasoning

# ── LLM MODE: Claude (Anthropic) ─────────────────────────────────────────────
# from langchain_anthropic import ChatAnthropic
# llm = ChatAnthropic(model="claude-haiku-4-5-20251001", temperature=0,
#                     api_key=settings.anthropic_api_key)

# ── LLM MODE: GPT-4o-mini (OpenAI) ───────────────────────────────────────────
# from langchain_openai import ChatOpenAI
# llm = ChatOpenAI(model="gpt-4o-mini", temperature=0,
#                  api_key=settings.openai_api_key)

mode = "LLM" if llm else "STUB"
print(f"Running in {mode} mode")

## 2. World Engine

The world engine is the ground truth the agents never see directly.

- 10×10 grid with forest (north), grassland (south), a rock ridge in the middle
- Fire ignited at cell (7, 2) — south-west grassland
- Hot, dry, south-west wind pushes fire toward the urban area in the south-east

In [ ]:
from ogar.world.scenarios.wildfire_basic import create_basic_wildfire
from ogar.world.grid import FireState, TerrainType

random.seed(42)   # reproducible fire spread

engine = create_basic_wildfire()

print(f"Grid: {engine.grid.rows}×{engine.grid.cols}")
print(f"Weather: {engine.weather.temperature_c}°C, "
      f"{engine.weather.humidity_pct}% humidity, "
      f"{engine.weather.wind_speed_mps} m/s wind")
print(f"Fire state: {engine.grid.summary()}")

In [ ]:
def render_grid(engine, sensor_positions=None):
    """ASCII render of the world grid — terrain, fire state, sensor locations."""
    # Terrain glyphs
    TERRAIN = {
        TerrainType.FOREST:    "T",
        TerrainType.GRASSLAND: ".",
        TerrainType.ROCK:      "#",
        TerrainType.WATER:     "~",
        TerrainType.SCRUB:     "s",
        TerrainType.URBAN:     "U",
    }
    sensor_positions = sensor_positions or set()

    rows = []
    for r in range(engine.grid.rows):
        row = []
        for c in range(engine.grid.cols):
            cell = engine.grid.get_cell(r, c)
            if cell.fire_state == FireState.BURNING:
                glyph = "F"
            elif cell.fire_state == FireState.BURNED:
                glyph = "*"
            elif (r, c) in sensor_positions:
                glyph = "S"
            else:
                glyph = TERRAIN.get(cell.terrain_type, "?")
            row.append(glyph)
        rows.append(" ".join(row))

    print("  " + " ".join(str(c) for c in range(engine.grid.cols)))
    for i, row in enumerate(rows):
        print(f"{i} {row}")
    print()
    print("Legend: T=Forest  .=Grass  #=Rock  ~=Water  s=Scrub  U=Urban  F=Burning  *=Burned  S=Sensor")

print("--- Initial world state ---")
render_grid(engine)

## 3. Sensors

Sensors sample from the world engine with Gaussian noise.  
Two clusters — north and south — each with temperature, smoke, wind, and humidity sensors.  
The agents see only sensor readings, never the grid directly.

In [ ]:
from ogar.sensors.world_sensors import (
    TemperatureSensor, SmokeSensor, WindSensor, HumiditySensor
)

sensors = [
    # ── Cluster North — covers the forest (rows 2-3) ──────────────────────────
    TemperatureSensor(source_id="temp-N1", cluster_id="cluster-north",
                      engine=engine, grid_row=2, grid_col=3, noise_std=0.5),
    TemperatureSensor(source_id="temp-N2", cluster_id="cluster-north",
                      engine=engine, grid_row=3, grid_col=6, noise_std=0.5),
    SmokeSensor(      source_id="smoke-N1", cluster_id="cluster-north",
                      engine=engine, grid_row=3, grid_col=4, noise_std=1.0),
    HumiditySensor(   source_id="hum-N1",   cluster_id="cluster-north",
                      engine=engine, noise_std=0.5),
    WindSensor(       source_id="wind-N1",  cluster_id="cluster-north",
                      engine=engine),

    # ── Cluster South — covers the grassland (rows 6-8), near the fire ────────
    TemperatureSensor(source_id="temp-S1", cluster_id="cluster-south",
                      engine=engine, grid_row=6, grid_col=2, noise_std=0.5),
    TemperatureSensor(source_id="temp-S2", cluster_id="cluster-south",
                      engine=engine, grid_row=7, grid_col=4, noise_std=0.5),
    SmokeSensor(      source_id="smoke-S1", cluster_id="cluster-south",
                      engine=engine, grid_row=7, grid_col=3, noise_std=1.0),
    HumiditySensor(   source_id="hum-S1",   cluster_id="cluster-south",
                      engine=engine, noise_std=0.5),
    WindSensor(       source_id="wind-S1",  cluster_id="cluster-south",
                      engine=engine),
]

# Sensor positions for grid rendering
sensor_positions = {
    (2,3),(3,6),(3,4),   # cluster north
    (6,2),(7,4),(7,3),   # cluster south
}

print(f"Created {len(sensors)} sensors across 2 clusters:")
for s in sensors:
    print(f"  {s.source_id:12s}  cluster={s.cluster_id}")

print()
print("--- World with sensor positions ---")
render_grid(engine, sensor_positions)

### Take a sample reading
One manual sensor read — shows what a raw `SensorEvent` looks like before it enters the pipeline.

In [ ]:
# Read one event from the temperature sensor closest to the ignition point
# emit() is synchronous — sensors sample the world engine directly
sample_event = sensors[5].emit()   # temp-S1, grid_row=6, grid_col=2

print("Raw SensorEvent:")
print(f"  event_id:    {sample_event.event_id}")
print(f"  source_id:   {sample_event.source_id}")
print(f"  source_type: {sample_event.source_type}")
print(f"  cluster_id:  {sample_event.cluster_id}")
print(f"  sim_tick:    {sample_event.sim_tick}")
print(f"  confidence:  {sample_event.confidence}")
print(f"  payload:     {sample_event.payload}")

## 4. Queue and Publisher

The `SensorPublisher` ticks all sensors in a loop and puts events on the queue.  
Setting `engine=engine` means it also calls `engine.tick()` before each sensor pass — world state and sensor readings stay in sync.

In [ ]:
from ogar.transport.queue import SensorEventQueue
from ogar.sensors.publisher import SensorPublisher

queue = SensorEventQueue(maxsize=500)

publisher = SensorPublisher(
    sensors=sensors,
    queue=queue,
    tick_interval_seconds=0.0,   # run as fast as possible (no wall-clock delay)
    engine=engine,               # auto-tick the world on each pass
)

print("Queue and publisher ready")

## 5. Agent Graphs + Shared Store

Build the cluster agent and supervisor with a **shared InMemoryStore**.

- Cluster agents write each `AnomalyFinding` to `("incidents", cluster_id)`
- Supervisor reads those incidents before assessing the situation
- Supervisor writes the situation summary to `("situations", "global")`

On K8s with Postgres, swap `InMemoryStore` for `AsyncPostgresStore` — one line change, everything else stays the same.

In [ ]:
from langgraph.store.memory import InMemoryStore
from ogar.agents.cluster.graph import build_cluster_agent_graph
from ogar.agents.supervisor.graph import build_supervisor_graph

# One shared store — cluster agents write, supervisor reads
# Swap for AsyncPostgresStore when Postgres is available
store = InMemoryStore()

cluster_graph   = build_cluster_agent_graph(llm=llm, store=store)
supervisor_graph = build_supervisor_graph(llm=llm, store=store)

print(f"Cluster agent:    {mode} mode  (store: {type(store).__name__})")
print(f"Supervisor agent: {mode} mode  (store: {type(store).__name__})")

## 6. Bridge Consumer

The bridge consumer reads from the queue and routes events to the cluster agent.  
It groups events by `cluster_id` and invokes the agent once per `batch_size` events.

In [ ]:
from ogar.bridge.consumer import EventBridgeConsumer

# All findings collected here as they arrive
findings_log = []

def on_finding(finding):
    findings_log.append(finding)
    # Show findings as they arrive
    confidence = finding['confidence']
    bar = "=" * int(confidence * 20)
    print(f"  [{finding['cluster_id']:15s}] [{bar:<20s}] {confidence:.0%}  "
          f"{finding['anomaly_type']}: {finding['summary'][:60]}")

consumer = EventBridgeConsumer(
    queue=queue,
    agent_graph=cluster_graph,
    on_finding=on_finding,
    batch_size=5,   # invoke the agent every 5 events per cluster
)

print("Consumer ready (batch_size=5)")

## 7. Run the Pipeline

Tick the world 20 times, emit sensor readings, and let the cluster agents process them.

In [ ]:
import langsmith

NUM_TICKS = 20

print("=" * 65)
print(f"Pipeline starting: {NUM_TICKS} world ticks")
print("=" * 65)
print()
print("Cluster findings as they arrive:")
print(f"  {'cluster':17s} {'confidence':22s} anomaly: summary")
print("  " + "-" * 62)

# langsmith.trace groups everything inside as one named run in the UI.
# You'll see the publisher run, all cluster agent invocations (as children),
# and any LLM/tool calls nested under each agent.
with langsmith.trace(
    name="ogar-pipeline",
    run_type="chain",
    metadata={"num_ticks": NUM_TICKS, "mode": mode, "clusters": ["cluster-north", "cluster-south"]},
):
    # 1. Run the publisher — ticks the world and fills the queue
    await publisher.run(ticks=NUM_TICKS)

    # 2. Drain the queue — runs the cluster agents
    await consumer.run(max_events=queue.total_enqueued)

print()
print("=" * 65)
print("Pipeline complete")
print(f"  World ticks:      {engine.current_tick}")
print(f"  Events produced:  {queue.total_enqueued}")
print(f"  Events consumed:  {consumer.events_consumed}")
print(f"  Agent invocations:{consumer.invocations}")
print(f"  Findings:         {len(findings_log)}")
print("=" * 65)

## 8. Ground Truth vs. Agent Findings

The key question: what *actually* happened in the world, and what did the agents detect?

In [ ]:
# ── Ground truth ─────────────────────────────────────────────────────────────
grid_summary = engine.grid.summary()
burning = [(r, c)
           for r in range(engine.grid.rows)
           for c in range(engine.grid.cols)
           if engine.grid.get_cell(r, c).fire_state == FireState.BURNING]
burned  = [(r, c)
           for r in range(engine.grid.rows)
           for c in range(engine.grid.cols)
           if engine.grid.get_cell(r, c).fire_state == FireState.BURNED]

print("GROUND TRUTH (world engine — agents never see this)")
print(f"  Currently burning: {len(burning)} cells  {burning}")
print(f"  Burned out:        {len(burned)} cells")
print(f"  Total affected:    {len(burning) + len(burned)} cells")
print()

print("--- World state after pipeline ---")
render_grid(engine, sensor_positions)

# ── What the agents reported ──────────────────────────────────────────────────
print("AGENT FINDINGS")
if not findings_log:
    print("  No anomalies detected.")
else:
    for f in findings_log:
        print(f"  [{f['cluster_id']:15s}] {f['anomaly_type']:20s} "
              f"conf={f['confidence']:.2f}")
        print(f"    {f['summary']}")
        print(f"    Sensors: {f['affected_sensors']}")

print()

# ── What's in the store ───────────────────────────────────────────────────────
print("CROSS-AGENT STORE CONTENTS")
for cluster_id in ["cluster-north", "cluster-south"]:
    items = store.search(("incidents", cluster_id))
    print(f"  ('incidents', '{cluster_id}')  →  {len(items)} item(s)")
    for item in items:
        v = item.value
        print(f"    [{item.key[:8]}]  {v.get('anomaly_type'):20s}  "
              f"conf={v.get('confidence', 0):.2f}  {v.get('summary', '')[:50]}")

## 9. Supervisor Agent

The supervisor receives all cluster findings via the **Send API fan-out**:  
it dispatches to N cluster agents in parallel, aggregates their results,  
assesses the overall situation, and decides what commands to issue.

In [ ]:
with langsmith.trace(
    name="ogar-supervisor",
    run_type="chain",
    metadata={"mode": mode, "findings_count": len(findings_log)},
):
    supervisor_result = supervisor_graph.invoke(
        {
            "active_cluster_ids": ["cluster-north", "cluster-south"],
            "cluster_findings":   findings_log,
            "messages":           [],
            "pending_commands":   [],
            "situation_summary":  None,
            "status":             "idle",
        },
        config={"run_name": "supervisor-assess-decide"},
    )

print("SUPERVISOR RESULT")
print(f"  Status:   {supervisor_result['status']}")
print(f"  Summary:  {supervisor_result.get('situation_summary', 'none')}")
print()

commands = supervisor_result.get("pending_commands", [])
print(f"  Commands issued: {len(commands)}")
for cmd in commands:
    print(f"    [{cmd.priority}] {cmd.command_type:12s} cluster={cmd.cluster_id}")
    print(f"         payload: {cmd.payload}")

## 10. What to Try Next

**Switch to LLM mode** — go back to cell 1 and uncomment the Claude or OpenAI lines.  
Re-run from cell 1. Watch the agent reasoning in the logs and in LangSmith.

**Change sensor positions** — move sensors away from the fire and see if the agents miss it.

**Increase `NUM_TICKS`** — let the fire spread further. Does the supervisor escalate differently?

**Change `batch_size`** — try `batch_size=1` for event-by-event processing vs `batch_size=10`.

**Add a second ignition** — after building the engine, call `engine.inject_ignition(row=2, col=8)`  
to start a second fire in the forest and see if the agents detect correlated events.